# RetinaNet（PASCAL VOC 2007を用いた物体検出）

---
## 目的
1段階（single-shot）の物体検出手法であるRetinaNetを，PASCAL VOC 2007データセットを用いて実装する．RetinaNetは，Feature Pyramid Network（FPN）による複数解像度の特徴マップと，多数のAnchor（アンカーボックス）を用いる点でSSDと同様の設計を持つが，最大の特徴は**Focal Loss**という損失関数にある．1段階検出器では，画像中のAnchorの大部分が「物体を含まない背景」であり，このクラス不均衡が学習を妨げる大きな要因となる．SSDはHard Negative Miningで負例を「間引く」ことでこの問題に対処したが，RetinaNetはFocal Lossにより，やさしい（簡単に識別できる）負例の損失を自動的に小さく重み付けすることで，Anchorを間引かずに同じ問題を解決する．本ノートブックでは，このFocal Lossの実装と，SSDとの違いに重点を置く．

## モジュールのインポートとGPUの確認
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
get_ipython().system('pip install -q torchmetrics')

import os
import math
import zipfile
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCDetection
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import box_iou, box_convert, nms
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込む．データセットの詳細（クラス構成やダウンロード方法）は`ssd.ipynb`と同じであるため，ここでは繰り返さない．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してほしい．

RetinaNetはFPNにより複数のストライド（8, 16, 32, 64）の特徴マップを使用するため，入力画像サイズはストライドの最小公倍数である64の倍数に設定する必要がある．本ノートブックでは`IMG_SIZE = 512`とする．

In [ ]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
# 背景（物体なし）を0番として，クラスラベルは1〜20を割り当てる
CLASS_TO_IDX = {name: i + 1 for i, name in enumerate(VOC_CLASSES)}
n_class = len(VOC_CLASSES) + 1  # 背景を含めたクラス数（分類ヘッドの設計は後述）
IMG_SIZE = 512

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)


def parse_voc_target(target):
    """VOCDetectionが返すXML由来のdictを，(boxes, labels)のTensorに変換する"""
    objects = target['annotation']['object']
    if isinstance(objects, dict):  # 物体が1つだけの場合はdictのまま返るため，リストに揃える
        objects = [objects]

    boxes, labels = [], []
    for obj in objects:
        bbox = obj['bndbox']
        boxes.append([float(bbox['xmin']), float(bbox['ymin']), float(bbox['xmax']), float(bbox['ymax'])])
        labels.append(CLASS_TO_IDX[obj['name']])

    return {'boxes': torch.tensor(boxes, dtype=torch.float32), 'labels': torch.tensor(labels, dtype=torch.long)}


class VOCDetectionDataset(torch.utils.data.Dataset):
    """VOCDetectionをラップし，画像のリサイズに合わせてボックス座標をスケールしたTensorを返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCDetection(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.to_tensor = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, target = self.voc[idx]
        w, h = image.size
        target = parse_voc_target(target)

        # 元画像サイズ -> (img_size, img_size) へのリサイズに合わせてボックス座標をスケール
        scale = torch.tensor([self.img_size / w, self.img_size / h, self.img_size / w, self.img_size / h])
        target['boxes'] = target['boxes'] * scale

        image = self.to_tensor(image)
        return image, target


def collate_fn(batch):
    # 画像1枚あたりの物体数が異なるため，ボックス・ラベルはリストのまま扱う
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


train_data = VOCDetectionDataset('trainval')
test_data = VOCDetectionDataset('test')

print(len(train_data), len(test_data))
print('クラス数（背景含む）:', n_class)

## RetinaNet：バックボーンとFeature Pyramid Network（FPN）
RetinaNetは，バックボーンネットワークの上に**Feature Pyramid Network（FPN）**と呼ばれるネットワーク構造を組み合わせ，その上に検出ヘッドを載せる構成を取る．FPNは，バックボーンの浅い層（高解像度・低レベルな特徴）と深い層（低解像度・高レベルな特徴）を，トップダウンパスと横方向結合（lateral connection）によって統合し，「どの解像度でも意味的に強い特徴」を持つ複数解像度の特徴マップ（P3, P4, P5, P6）を作る．

具体的には，ResNet50の`layer2`・`layer3`・`layer4`の出力（C3, C4, C5，ストライド8, 16, 32）それぞれに1×1畳み込み（lateral connection）でチャンネル数を256に揃えたのち，深い特徴マップを2倍にアップサンプリングして浅い特徴マップに加算する（トップダウンパス）．加算後の特徴マップにはエイリアシングを抑えるため3×3畳み込み（smoothing）を適用し，それぞれP3, P4, P5とする．さらにC5に3×3のストライド2畳み込みを適用し，ストライド64のP6を作る（原論文ではP6, P7の2つを追加するが，本ノートブックでは実装を単純化しP6のみを追加し，P3〜P6の4段階のピラミッドとする）．

In [ ]:
class FPNNeck(nn.Module):
    """ResNetのC3, C4, C5から，トップダウンパス＋横方向結合でP3〜P6を作るFPN"""
    def __init__(self, in_channels_list=(512, 1024, 2048), out_channels=256):
        super().__init__()
        c3_ch, c4_ch, c5_ch = in_channels_list
        self.lateral3 = nn.Conv2d(c3_ch, out_channels, kernel_size=1)
        self.lateral4 = nn.Conv2d(c4_ch, out_channels, kernel_size=1)
        self.lateral5 = nn.Conv2d(c5_ch, out_channels, kernel_size=1)

        self.smooth3 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth4 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth5 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        self.p6 = nn.Conv2d(c5_ch, out_channels, kernel_size=3, stride=2, padding=1)  # C5からストライド2でP6（ストライド64）を作る

    def forward(self, c3, c4, c5):
        p5 = self.lateral5(c5)
        p4 = self.lateral4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode='nearest')
        p3 = self.lateral3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode='nearest')

        p3 = self.smooth3(p3)
        p4 = self.smooth4(p4)
        p5 = self.smooth5(p5)
        p6 = self.p6(c5)

        return [p3, p4, p5, p6]  # ストライド 8, 16, 32, 64


class RetinaNetBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool, resnet.layer1)
        self.layer2 = resnet.layer2  # C3, stride  8, channels  512
        self.layer3 = resnet.layer3  # C4, stride 16, channels 1024
        self.layer4 = resnet.layer4  # C5, stride 32, channels 2048
        self.fpn = FPNNeck(in_channels_list=(512, 1024, 2048), out_channels=256)

    def forward(self, x):
        x = self.stem(x)
        c3 = self.layer2(x)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        return self.fpn(c3, c4, c5)  # [P3, P4, P5, P6]


FPN_STRIDES = [8, 16, 32, 64]
FPN_OUT_CHANNELS = 256

## Anchor（アンカーボックス）の生成
SSDのデフォルトボックスと同様に，各特徴マップの各セルに複数サイズ・アスペクト比のAnchorをあらかじめ格子状に配置しておく．RetinaNetの原論文では，各ピラミッドレベルに「3種類のスケール（$2^0, 2^{1/3}, 2^{2/3}$）× 3種類のアスペクト比（$1{:}2, 1{:}1, 2{:}1$）」の合計9種類のAnchorを配置する．

各レベルのAnchorの基準サイズ（面積の一辺）は，そのレベルのストライドの4倍（P3: 32, P4: 64, P5: 128, P6: 256）とする．これはP3のAnchorが32×32画素相当の小さな物体を，P6のAnchorが256×256画素相当の大きな物体を担当するように設計されていることを意味する．SSDでは正規化座標（0〜1）でデフォルトボックスを扱ったが，本ノートブックではFPNの各レベルの実際の画素座標を直接使う方が実装がシンプルになるため，Anchorはピクセル座標（`cx, cy, w, h`）で表現する．

In [ ]:
ANCHOR_SCALES = [2 ** 0, 2 ** (1 / 3), 2 ** (2 / 3)]
ANCHOR_RATIOS = [0.5, 1.0, 2.0]  # h / w
num_anchors_per_cell = len(ANCHOR_SCALES) * len(ANCHOR_RATIOS)  # 9


def generate_anchors():
    anchors = []
    for stride in FPN_STRIDES:
        f = IMG_SIZE // stride
        base_size = stride * 4  # レベルごとの基準Anchorサイズ（P3:32, P4:64, P5:128, P6:256）

        for i in range(f):
            for j in range(f):
                cx = (j + 0.5) * stride
                cy = (i + 0.5) * stride

                for scale in ANCHOR_SCALES:
                    for ratio in ANCHOR_RATIOS:
                        area = (base_size * scale) ** 2
                        w = math.sqrt(area / ratio)
                        h = math.sqrt(area * ratio)
                        anchors.append([cx, cy, w, h])

    return torch.tensor(anchors, dtype=torch.float32)


anchors = generate_anchors().to(device)  # (num_anchors, 4), ピクセル座標の (cx, cy, w, h)
anchors_xyxy = box_convert(anchors, 'cxcywh', 'xyxy').clamp(0, IMG_SIZE)
print('Anchorの総数:', anchors.shape[0])

## Anchorのエンコード・デコードとGTとのマッチング
学習時には，正解ボックス（GT）を直接回帰させるのではなく，マッチしたAnchorからの相対的なオフセットに変換（エンコード）して学習する．推論時には，ネットワークが出力したオフセットを対応するAnchorに適用して画像上の座標に戻す（デコード）．基本的な考え方は`ssd.ipynb`のデフォルトボックスのエンコード・デコードと同じだが，本ノートブックではSSDで用いた`variances`によるスケール調整は行わず，Faster R-CNN系の検出器で広く使われる素朴なパラメータ化（$t_x=(g_x-a_x)/a_w$ など）を用いる．

Anchorと GTのマッチングも，考え方はSSDと同様に「各Anchorに最もIoUが高いGTを割り当てる」方式だが，RetinaNetでは一般に次の3値のラベルを使う．

- IoU ≥ 0.5 のAnchor：そのGTのクラスを教師信号とする正例
- IoU < 0.4 のAnchor：背景（クラス0）とする負例
- 0.4 ≤ IoU < 0.5 のAnchor：どちらとも判定しにくいためAnchorを無視（損失計算から除外）する

さらにSSDと同様，各GTボックスに対してIoUが最大となるAnchorは，上記の閾値によらず必ず正例として割り当てる．

In [ ]:
def encode_boxes(matched_boxes_xyxy):
    """マッチしたGTボックス（xyxy, ピクセル座標）を，Anchorからのオフセットにエンコードする"""
    matched_cxcywh = box_convert(matched_boxes_xyxy, 'xyxy', 'cxcywh')
    txy = (matched_cxcywh[:, :2] - anchors[:, :2]) / anchors[:, 2:]
    twh = torch.log(matched_cxcywh[:, 2:] / anchors[:, 2:] + 1e-8)
    return torch.cat([txy, twh], dim=1)


def decode_boxes(loc_preds):
    """ネットワークが出力したオフセットを，画像上のボックス座標（xyxy, ピクセル座標）にデコードする"""
    cxcy = loc_preds[..., :2] * anchors[:, 2:] + anchors[:, :2]
    wh = torch.exp(loc_preds[..., 2:]) * anchors[:, 2:]
    boxes_cxcywh = torch.cat([cxcy, wh], dim=-1)
    return box_convert(boxes_cxcywh, 'cxcywh', 'xyxy').clamp(min=0, max=IMG_SIZE)


def match_anchors(gt_boxes, gt_labels, pos_iou=0.5, neg_iou=0.4):
    """1枚の画像内のGTボックスをAnchorに割り当て，学習用のラベル・オフセットターゲットを作成する"""
    num_anchors_total = anchors.shape[0]
    if gt_boxes.shape[0] == 0:
        return torch.zeros(num_anchors_total, dtype=torch.long, device=device), torch.zeros(num_anchors_total, 4, device=device)

    ious = box_iou(gt_boxes, anchors_xyxy)  # (num_gt, num_anchors)

    best_gt_iou, best_gt_idx = ious.max(dim=0)  # 各Anchorに対して最もIoUが高いGT
    _, best_anchor_idx = ious.max(dim=1)  # 各GTに対して最もIoUが高いAnchorは，閾値によらず必ず正例にする
    best_gt_iou[best_anchor_idx] = 1.0
    best_gt_idx[best_anchor_idx] = torch.arange(gt_boxes.shape[0], device=gt_boxes.device)

    labels = gt_labels[best_gt_idx].clone()
    labels[best_gt_iou < pos_iou] = 0  # 背景（負例）
    ignore_mask = (best_gt_iou >= neg_iou) & (best_gt_iou < pos_iou)
    labels[ignore_mask] = -1  # 損失計算から除外するAnchor

    matched_boxes = gt_boxes[best_gt_idx]
    loc_targets = encode_boxes(matched_boxes)

    return labels, loc_targets

## 検出ヘッド（分類・ボックス回帰）
RetinaNetの検出ヘッドは，SSDとは異なり**全てのピラミッドレベル（P3〜P6）で重みを共有する**小さな畳み込みサブネットワークである（SSDの`SSDHead`は，特徴マップごとに別々の`nn.Conv2d`を`ModuleList`として持ち，レベル間で重みを共有していなかった）．具体的には，4層の3×3畳み込み（チャンネル数256，各層の後にReLU）を共通の特徴抽出部として使い，その後に分類用・ボックス回帰用それぞれ別の最終畳み込み層を1つずつ適用する．同じ`RetinaNetHead`のインスタンスをP3〜P6すべてに適用することで，レベルによらない共通の特徴表現で検出を行う．

分類ヘッドの出力チャンネル数は`num_anchors_per_cell * (n_class - 1)`とする．RetinaNetは，通常のSSD/Faster R-CNNのような「背景を含む`n_class`クラスのSoftmax」ではなく，**背景クラスを明示的に持たず，前景の各クラスを独立した2値分類器（シグモイド）として扱う**設計を取る（`n_class - 1`個の1 vs 全のバイナリ分類器）．これはFocal Lossの定式化（後述）が本来2値分類に対する損失として定義されているためであり，全クラスのスコアが低ければ暗黙的に背景として扱われる．

また，分類ヘッドの最終層のバイアスは，学習開始時に「ほとんどのAnchorが背景である」という事前知識を反映するため，$-\log((1-\pi)/\pi)$（$\pi=0.01$）で初期化する．こうすることで，学習初期は各Anchorの前景スコアが低い値からスタートし，Focal Lossの値が発散せず安定する（原論文で述べられている実装上の工夫）．

In [ ]:
class RetinaNetHead(nn.Module):
    def __init__(self, in_channels, num_anchors_per_cell, n_class, prior_prob=0.01):
        super().__init__()
        n_fg_class = n_class - 1  # 背景を明示的なクラスとして持たない

        self.cls_subnet = nn.Sequential(
            *sum([[nn.Conv2d(in_channels, in_channels, 3, padding=1), nn.ReLU(inplace=True)] for _ in range(4)], []))
        self.box_subnet = nn.Sequential(
            *sum([[nn.Conv2d(in_channels, in_channels, 3, padding=1), nn.ReLU(inplace=True)] for _ in range(4)], []))

        self.cls_out = nn.Conv2d(in_channels, num_anchors_per_cell * n_fg_class, kernel_size=3, padding=1)
        self.box_out = nn.Conv2d(in_channels, num_anchors_per_cell * 4, kernel_size=3, padding=1)

        # 学習初期の安定化のため，分類ヘッドのバイアスを背景寄りに初期化する
        bias_value = -math.log((1 - prior_prob) / prior_prob)
        nn.init.constant_(self.cls_out.bias, bias_value)

        self.n_fg_class = n_fg_class

    def forward(self, features):
        """featuresはP3〜P6のリスト．全レベルで同じ重み（self.cls_subnetなど）を使い回す"""
        cls_outputs, box_outputs = [], []
        for feat in features:
            cls_feat = self.cls_subnet(feat)
            box_feat = self.box_subnet(feat)

            cls_out = self.cls_out(cls_feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, self.n_fg_class)
            box_out = self.box_out(box_feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, 4)

            cls_outputs.append(cls_out)
            box_outputs.append(box_out)

        return torch.cat(cls_outputs, dim=1), torch.cat(box_outputs, dim=1)

## 損失関数：Focal Loss
1段階検出器最大の課題は，1枚の画像あたり数万個のAnchorのうち，物体を含む正例はごく少数（多くて数十個）で，残りの大部分が背景の負例であるという極端なクラス不均衡である．通常のクロスエントロピー損失をそのまま使うと，圧倒的多数の「簡単に識別できる」背景の負例による損失の合計が，少数の正例による損失を圧倒し，学習が非効率になる．

SSDはこの問題を，Hard Negative Miningによって**負例の大部分を損失計算から除外する**ことで対処した（正例の`neg_pos_ratio`倍だけを残す）．一方RetinaNetは，Anchorを間引くのではなく，**全てのAnchorを使いながら，各Anchorの損失を「識別の簡単さ」に応じて動的に重み付けする**という別のアプローチでこの問題を解決する．これがFocal Lossである．

2値分類に対するFocal Lossは，正解クラスに対するモデルの推定確率を$p_t$として，次のように定義される．

$$FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

- $(1-p_t)^\gamma$：**modulating factor**．$p_t$が1に近い（正しく，かつ自信を持って識別できている簡単な例）ほど$(1-p_t)^\gamma$は0に近づき，損失への寄与が小さくなる．逆に$p_t$が小さい（誤って識別している難しい例）場合は$(1-p_t)^\gamma$はほぼ1のままで，通常のクロスエントロピーとほぼ同じ大きさの損失が課される．$\gamma$（本ノートブックでは2.0）はこの重み付けの強さを調整するハイパーパラメータで，$\gamma=0$のとき通常のクロスエントロピーに一致する．
- $\alpha_t$：前景・背景間のクラス数不均衡を補正するための重み．前景クラスには大きい値（本ノートブックでは0.25），背景には$1-\alpha_t$を用いる（注：modulating factorだけでも簡単な背景の損失は小さくなるが，背景Anchorの総数自体が膨大なため，$\alpha$による重み付けも併用することで前景・背景間のバランスをさらに調整する）．

Focal Lossは，簡単な背景の負例を「捨てる」のではなく「小さく重み付けする」ため，Hard Negative Miningのような追加のサンプリング処理が不要になる．本ノートブックの学習ループでは，SSDのようにAnchorを間引かず，**正例・負例を含む全てのAnchor**（マッチング時に無視と判定されたAnchorを除く）を使って損失を計算する．

In [ ]:
class FocalLoss(nn.Module):
    """前景クラスごとの独立な2値分類として，Anchorのクラス分類にFocal Lossを適用する"""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, cls_preds, cls_targets):
        """
        cls_preds: (B, num_anchors, n_fg_class) 生のロジット
        cls_targets: (B, num_anchors) 0=背景, 1..n_fg_class=前景クラス, -1=無視するAnchor
        戻り値: 有効なAnchor・クラスに対するFocal Lossの合計（正規化は呼び出し側で行う）
        """
        n_fg_class = cls_preds.size(-1)
        valid_mask = cls_targets != -1
        pos_mask = cls_targets > 0

        # 前景クラスをone-hotベクトルに変換（背景・無視Anchorは全チャンネル0のまま）
        targets_onehot = torch.zeros_like(cls_preds)
        targets_onehot[pos_mask] = F.one_hot(cls_targets[pos_mask] - 1, num_classes=n_fg_class).float()

        p = torch.sigmoid(cls_preds)
        ce = F.binary_cross_entropy_with_logits(cls_preds, targets_onehot, reduction='none')
        p_t = p * targets_onehot + (1 - p) * (1 - targets_onehot)
        alpha_t = self.alpha * targets_onehot + (1 - self.alpha) * (1 - targets_onehot)
        loss = alpha_t * (1 - p_t) ** self.gamma * ce  # (B, num_anchors, n_fg_class)

        return loss[valid_mask].sum()


class RetinaNetLoss(nn.Module):
    """Focal Loss（分類）とSmooth L1 Loss（ボックス回帰，正例のみ）を組み合わせた損失"""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.focal_loss = FocalLoss(alpha=alpha, gamma=gamma)
        self.smooth_l1 = nn.SmoothL1Loss(reduction='sum')

    def forward(self, cls_preds, box_preds, cls_targets, box_targets):
        pos_mask = cls_targets > 0
        num_pos = pos_mask.sum().clamp(min=1)

        cls_loss = self.focal_loss(cls_preds, cls_targets) / num_pos
        box_loss = self.smooth_l1(box_preds[pos_mask], box_targets[pos_mask]) / num_pos

        return cls_loss + box_loss

## ネットワークの作成
バックボーン＋FPN（`RetinaNetBackbone`），検出ヘッド（`RetinaNetHead`）を組み合わせて`RetinaNet`モデルを構築する．`.to(device)`によりモデルのパラメータを`device`上に配置し，最適化手法にはSSDと同様にモーメンタムSGD（学習率0.001，モーメンタム0.9，weight decay 5e-4）を用いる．

In [ ]:
class RetinaNet(nn.Module):
    def __init__(self, n_class, num_anchors_per_cell):
        super().__init__()
        self.backbone = RetinaNetBackbone()
        self.head = RetinaNetHead(FPN_OUT_CHANNELS, num_anchors_per_cell, n_class)

    def forward(self, x):
        features = self.backbone(x)  # [P3, P4, P5, P6]
        return self.head(features)


model = RetinaNet(n_class=n_class, num_anchors_per_cell=num_anchors_per_cell)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)
criterion = RetinaNetLoss(alpha=0.25, gamma=2.0)

n_params = sum(p.numel() for p in model.parameters())
print('総パラメータ数:', n_params)

## 学習
読み込んだVOC2007データセットと作成したネットワークを用いて学習を行う．ミニバッチサイズを8，学習エポック数を20とする．画像1枚あたりの物体数が異なるため，`collate_fn`でバッチを作成し，各画像のGTボックスをバッチ内でループしながらAnchorとマッチングさせる．

In [ ]:
batch_size = 8
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0

    for images, targets in train_loader:
        images = images.to(device)

        batch_labels, batch_box_targets = [], []
        for t in targets:
            labels, box_targets = match_anchors(t['boxes'].to(device), t['labels'].to(device))
            batch_labels.append(labels)
            batch_box_targets.append(box_targets)
        batch_labels = torch.stack(batch_labels)
        batch_box_targets = torch.stack(batch_box_targets)

        cls_preds, box_preds = model(images)

        loss = criterion(cls_preds, box_preds, batch_labels, batch_box_targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    print("epoch: {}, mean loss: {}, elapsed time: {}".format(epoch, sum_loss / len(train_loader), time() - train_start))

## 推論（デコード・NMS）
学習したモデルで推論を行うための関数を定義する．分類ヘッドの出力はSSDのSoftmaxではなくシグモイドでクラスごとの独立な確率に変換し，信頼度が閾値を超えるAnchorをデコードしたのち，`torchvision.ops.nms`でクラスごとに重複した検出結果を除去（Non-Maximum Suppression）する．

In [ ]:
def predict(model, image_tensor, conf_threshold=0.5, nms_threshold=0.45):
    model.eval()
    with torch.no_grad():
        cls_preds, box_preds = model(image_tensor.unsqueeze(0).to(device))
    scores_all = torch.sigmoid(cls_preds[0])  # (num_anchors, n_fg_class)
    boxes_all = decode_boxes(box_preds[0])

    result_boxes, result_scores, result_labels = [], [], []
    for class_idx in range(1, n_class):  # 前景クラス（0=背景は暗黙のため対象外）
        class_scores = scores_all[:, class_idx - 1]
        mask = class_scores > conf_threshold
        if mask.sum() == 0:
            continue
        class_boxes = boxes_all[mask]
        class_scores = class_scores[mask]

        keep = nms(class_boxes, class_scores, nms_threshold)
        result_boxes.append(class_boxes[keep])
        result_scores.append(class_scores[keep])
        result_labels.append(torch.full((len(keep),), class_idx, dtype=torch.long))

    if len(result_boxes) == 0:
        return torch.zeros(0, 4), torch.zeros(0), torch.zeros(0, dtype=torch.long)

    return torch.cat(result_boxes).cpu(), torch.cat(result_scores).cpu(), torch.cat(result_labels).cpu()

## テスト
評価指標には`ssd.ipynb`と同様，`torchmetrics`の`MeanAveragePrecision`を用いてmAPを算出する（実装の詳細は`ssd.ipynb`を参照）．VOCで伝統的に使われるIoU閾値0.5でのmAP（`map_50`）を確認する．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False, collate_fn=collate_fn)

metric = MeanAveragePrecision(iou_type='bbox')
model.eval()

for images, targets in test_loader:
    boxes, scores, labels = predict(model, images[0])

    preds = [{'boxes': boxes, 'scores': scores, 'labels': labels}]
    gts = [{'boxes': targets[0]['boxes'], 'labels': targets[0]['labels']}]
    metric.update(preds, gts)

result = metric.compute()
print('mAP@0.5:', result['map_50'].item())
print('mAP@[0.5:0.95]:', result['map'].item())

## 検出結果の可視化
テストデータ1枚に対する検出結果を，`torchvision.utils.draw_bounding_boxes`を用いて画像上に描画する．

In [ ]:
image, target = test_data[0]
boxes, scores, labels = predict(model, image)

# 正規化を戻して0-255のuint8画像に変換
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_show = (image * std + mean).clamp(0, 1)
img_show = (img_show * 255).to(torch.uint8)

label_names = [f'{VOC_CLASSES[l - 1]}: {s:.2f}' for l, s in zip(labels, scores)]
img_with_boxes = draw_bounding_boxes(img_show, boxes, labels=label_names, colors='red', width=2)

plt.figure(figsize=(6, 6))
plt.imshow(img_with_boxes.permute(1, 2, 0))
plt.axis('off')
plt.show()

## オリジナルRetinaNetとの違い
本ノートブックで実装したRetinaNetは，原論文の構成を一部簡略化している．主な違いは次の通りである．

| | 原論文 | 本ノートブック |
|---|---|---|
| バックボーン | ResNet-50-FPN（もしくはResNet-101-FPN） | ResNet50（ImageNet事前学習済み）＋FPN（変更なし） |
| 入力画像サイズ | 短辺800（複数スケール学習あり） | 512×512固定 |
| ピラミッドレベル | P3〜P7（5段階） | P3〜P6（4段階，P7を省略） |
| Anchor | レベルごとに3スケール×3アスペクト比＝9種類 | 同左（変更なし） |
| 分類ヘッドの設計 | 背景クラスを持たない前景クラスごとの2値分類（シグモイド） | 同左（変更なし） |
| 損失関数 | Focal Loss（分類）＋Smooth L1（回帰） | 同左（変更なし） |
| NMS・IoU計算 | 独自実装 | `torchvision.ops`の`nms`・`box_iou`を使用 |
| 学習スケジュール | 大規模データでの長時間学習（複数GPU，多数epoch） | 単一GPU，epoch数を縮小 |

RetinaNetの核心であるFocal Lossおよび検出ヘッドの重み共有という設計は原論文通りに実装しており，主な簡略化はピラミッドレベル数と入力解像度，学習スケジュールである．

## 課題

1. Focal Lossの`gamma`（フォーカシングパラメータ）を0（通常のクロスエントロピーに相当）や1，5など複数の値に変更して学習し，収束の速さや損失の推移がどのように変化するか確認しましょう．

2. Focal Lossの`alpha`を0.25以外の値（例えば0.5や0.75）に変更し，前景・背景間のバランス調整が学習にどのように影響するか確認しましょう．

3. `FocalLoss`を，前景・背景に関わらず重み付けを行わない（$\alpha_t=1$固定，modulating factorも使わない）通常のクロスエントロピー損失に置き換えて学習し，Hard Negative Miningなしでクラス不均衡を放置した場合に学習がどのように不安定化するか（あるいは背景クラスに引っ張られるか）を確認しましょう．